# 03 — The Measurement Loop

In 02 we saw how to **use evaluation metrics** and **understand tradeoffs** (e.g. recall vs precision). Here we make the **habit** explicit: **measure → change one thing → measure again → compare.** Don't guess whether a change helps—run it, evaluate against ground truth, and compare. We use the same entity resolution data as 02 so the numbers are comparable; you'll use this same loop for fuzzy thresholds, blocking keys, and blend cutoffs.

**Back:** [02 Exact Matching](02_exact_matching.ipynb) · **Next:** [04 Fuzzy Matching](04_fuzzy_matching.ipynb)

## 1. The loop

1. Run matcher with your current setup (rules, and later: blocking, fuzzy, blend).
2. Evaluate against ground truth: `results.evaluate(ground_truth, left_id_col="id", right_id_col="id_right")`.
3. Change **one** thing (e.g. add a rule, try a different blocking key, raise or lower a threshold).
4. Re-run and evaluate again.
5. Compare. Did precision or recall improve? Your data decides.

Repeat until you're satisfied. No magic—just measure, tune, compare.

## 2. What the metrics mean

Before we run the loop, here's what you're reading when you look at the table.

- **Precision:** Of the pairs the matcher *said* are matches, how many are actually in ground truth? High precision = few false positives (we're not linking the wrong people).

- **Recall:** Of *all* the true pairs in ground truth, how many did we find? High recall = we didn't miss many (we're not leaving true pairs undiscovered).

- **F1:** A single number that balances precision and recall (harmonic mean). Useful when you want to compare strategies at a glance without choosing one of the two.

You can't always maximise both: adding rules or lowering thresholds often boosts recall but can hurt precision, and vice versa. The loop helps you see that tradeoff in your data.

## 3. Load data and ground truth

Same data as 02: load **raw**, **standardize**, then run the loop. We use the same ground truth (30 pairs) so every run is comparable.

In [1]:
import sys
from pathlib import Path
import polars as pl
from matcher import Matcher

_tutorial = Path.cwd() / "docs" / "tutorial"
if not (_tutorial / "tutorial_data").exists():
    _tutorial = (
        Path.cwd()
        if (Path.cwd() / "tutorial_data").exists()
        else Path.cwd().parent.parent / "docs" / "tutorial"
    )
sys.path.insert(0, str(_tutorial))
from tutorial_data import load_entity_resolution, standardize_for_matching

left_raw, right_raw, ground_truth = load_entity_resolution()
left, right = standardize_for_matching(left_raw, right_raw)
matcher = Matcher(
    left=left,
    right=right,
    left_id="id",
    right_id="id",
)
n_gt = ground_truth.shape[0]
print(f"Ground truth: {n_gt} pairs. We'll compare every run against this.")

Ground truth: 30 pairs. We'll compare every run against this.


## 4. Run A: one rule

First run: `full_name` only. Run, evaluate, note the numbers.

In [2]:
results_a = matcher.match(on="full_name")
metrics_a = results_a.evaluate(
    ground_truth,
    left_id_col="id",
    right_id_col="id_right",
)
print(f"Run A: {results_a.count} matches")
print(f"Recall: {metrics_a['recall']:.2%}")

Run A: 10 matches
Recall: 33.33%


## 5. Run B: change one thing

Change **one** thing: add a cascading rule (`full_name`, then `email_clean` for unmatched). Re-run, evaluate again.

In [3]:
results_b = matcher.match(on=[
    "full_name",
    ["email_clean"],
])
metrics_b = results_b.evaluate(
    ground_truth,
    left_id_col="id",
    right_id_col="id_right",
)
print(f"Run B: {results_b.count} matches")
print(f"Recall: {metrics_b['recall']:.2%}")

Run B: 20 matches
Recall: 66.67%


## 6. Run C: full_name then zip_code (cascade)

We **choose zip_code** as the second rule to show how the results change when you add another rule. From reality we know zip is a **bad choice** for matching—many people share a zip—so the zip cascade finds more true pairs but also huge numbers of false ones. The effect here is dramatic; the idea is the same in your data: run it, put it in the table, and see how recall and precision move. Compare to A (full_name only) and B (full_name + email).

In [4]:
results_c = matcher.match(on=[
    "full_name",
    ["zip_code"],
])
metrics_c = results_c.evaluate(
    ground_truth,
    left_id_col="id",
    right_id_col="id_right",
)
print(f"Run C: {results_c.count} matches")
print(f"Recall: {metrics_c['recall']:.2%}")

Run C: 48419 matches
Recall: 66.67%


Run C returns **48,419** pairs (vs 20 for B) with the same recall. How? The cascade runs the second rule only on *unmatched* left rows (~490 after full_name). Each of those is then matched to *every* right row with the same zip. The data has only **5 zip values**, so ~100 right rows per zip—so we get roughly 490 × 100 ≈ 49,000 pairs from the zip step alone (plus the 10 from full_name). That's well under 500×500; the explosion is "one left row → many right rows per zip," not a full cross join. Almost all are false positives, so precision collapses in the table below.

## 7. Compare

Same ground truth, three strategies: full_name only (A), full_name + email (B), full_name then zip (C). We used zip in C to show how the numbers change when you add a rule—and because in reality zip is a bad choice (many people share one), the effect is dramatic. Put the metrics side by side and see how each cascade changes recall and precision.

In [ ]:
comparison = pl.DataFrame({
    "run": [
        "A (full_name only)",
        "B (full_name + email_clean)",
        "C (full_name + zip_code)",
    ],
    "precision": [
        f"{metrics_a['precision']:.2%}",
        f"{metrics_b['precision']:.2%}",
        f"{metrics_c['precision']:.2%}",
    ],
    "recall": [
        f"{metrics_a['recall']:.2%}",
        f"{metrics_b['recall']:.2%}",
        f"{metrics_c['recall']:.2%}",
    ],
    "f1": [
        f"{metrics_a['f1']:.2%}",
        f"{metrics_b['f1']:.2%}",
        f"{metrics_c['f1']:.2%}",
    ],
})
display(comparison)

run,precision,recall,f1
str,str,str,str
"""A (full_name only)""","""100.00%""","""33.33%""","""50.00%"""
"""B (full_name + email_clean)""","""100.00%""","""66.67%""","""80.00%"""
"""C (full_name + zip_code)""","""0.04%""","""66.67%""","""0.08%"""


**What the table is telling you:** **A** finds only the pairs with identical names (33.33% recall) and nothing else (100% precision). **B** adds pairs that match on email but not on exact name (e.g. William vs Bill)—recall doubles, precision stays high. **C** adds a zip cascade: we chose zip to show how results can change when you add a rule. From reality we know zip is a bad choice—many people share a zip—so precision collapses; the effect here is dramatic. The lesson is the same in your data: the table shows how recall and precision move. Next: how to use this when you tune.

## 8. How to respond: using the metrics when you tune

Once you have a table like the one above, use it to decide what to change next—not by guessing, but by reading the numbers.

**Read the table first.**  
- If **recall** is too low, you're missing true pairs. The fix is usually to *add* signal: another rule or cascade, a *lower* fuzzy threshold, or *relax* a constraint. Re-run, put the new row in the table, and see if recall went up without destroying precision.  
- If **precision** is too low, you're linking too many non-matches. The fix is usually to *tighten*: fewer or stricter rules, a *higher* fuzzy threshold, or a more discriminating key (e.g. avoid a blunt zip-only step). Re-run and compare.  
- Don't optimise one number in isolation. Improving recall by adding a rule that tanks precision (like C here) is only useful if you're prepared to clean up the false positives later. Your **use case** sets the target: e.g. deduplication often favours precision; broad linkage might favour recall.

**The habit.**  
Every time you change one thing (rules, threshold, blocking key, blend), run the matcher, evaluate against ground truth, and add a row or run to the comparison table. Then decide whether the change was worth it. You'll use this same loop for exact rules, fuzzy thresholds, blocking, and blend cutoffs.

**Leading into fuzzy.**  
Run A (full_name only) found just the pairs with *identical* names. We're still missing true pairs where the name is a variant—Jon vs Jonathan, Katherine vs Catherine, typos. Next we add **fuzzy** matching on the name so similar strings can match, and we use this same measurement loop to choose a threshold.

**Next:** [04 Fuzzy Matching](04_fuzzy_matching.ipynb) — fuzzy on name to recover those pairs, and tune the threshold with the same loop.